# Chapter 30 — Dashboard design: KPIs, filters, drill-down, and synchronized views

**Book role:** Visualization  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Design an analytical dashboard around a decision or question.
- Calculate KPIs with documented definitions.
- Synchronize filters across cards, table, and chart.
- Provide sensible empty, loading, and stale-data states.

## 1. Start with the decision, not the chart catalog

Before placing charts, name the decision the user needs to make. A sales manager might need to see whether sales are meeting target, which regions drive variance, and which products explain the change. Select a small number of metrics and visualizations that answer those questions. A dashboard filled with every available chart increases cognitive load without necessarily increasing insight.

Use visual hierarchy: headline KPIs, a main trend/comparison, detail breakdown, and a table for follow-up. Display comparison context (previous period, target, or baseline) only when the periods and denominator are comparable. Use consistent date filters and communicate if each chart uses a different grain or subset.

### Worked example 1: Calculate a KPI summary with defined semantics

The `unique_orders` metric demonstrates why row count and transaction count are not interchangeable. Margin is undefined when revenue is zero; represent that state deliberately instead of displaying an unexplained infinity.

In [ ]:
import pandas as pd

def sales_kpis(frame: pd.DataFrame) -> dict:
    required = {"order_id", "revenue", "profit"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Required columns: {sorted(required)}")
    row_count = len(frame)
    revenue = float(frame["revenue"].sum())
    profit = float(frame["profit"].sum())
    return {
        "line_count": row_count,
        "unique_orders": int(frame["order_id"].nunique()),
        "revenue": revenue,
        "profit": profit,
        "profit_margin": profit / revenue if revenue else None,
    }

sample = pd.DataFrame({"order_id": [1, 1, 2], "revenue": [100, 50, 80], "profit": [20, 10, 8]})
print(sales_kpis(sample))

## 2. One source of truth for filters

A region selector should drive the headline totals, chart, and table coherently. If each view reads its own widgets or runs a differently interpreted query, users can see mismatched totals. Create one filter-state object; when the Apply button is pressed, validate it, request the result through the service, and update all views from one result or set of clearly related results.

Some metrics need separate query plans—for example, an overall target may not be available at the transaction grain. In those cases, document the distinct data source and join/aggregation semantics rather than forcing every view to share a single DataFrame. Synchronization means consistent user intent, not necessarily an identical underlying query.

### Worked example 2: Keep drill-down state explicit

The state object makes drill-down and reset logic testable. A GUI controller can update controls to reflect this state and re-run the data service.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DashboardState:
    start_date: str
    end_date: str
    region: str = "All"
    category: str = "All"
    status: str = "Paid"


def with_region(state: DashboardState, selected_region: str) -> DashboardState:
    return DashboardState(
        start_date=state.start_date,
        end_date=state.end_date,
        region=selected_region,
        category=state.category,
        status=state.status,
    )

state = DashboardState("2026-01-01", "2027-01-01")
print(with_region(state, "North"))

## 3. Refresh and stale data semantics

A dashboard should display when its data was refreshed and whether the current results are live, cached, or based on a static file. If refresh fails, decide whether to keep the previous successful data with a stale indicator or clear the display. Silently replacing a valid dataset with empty data after a transient failure is often worse than preserving the prior results and stating that refresh failed.

Drill-down should preserve context. Selecting a bar for “North” can apply a North-region filter to the table, but the app must provide a visible way back and show the active filters. Include a reset action, avoid hidden state changes, and ensure exports reflect the exact filter state displayed.

## Guided lab: Lab: executive dashboard

Build a dashboard with at least four KPIs, a time-trend chart, a category comparison, a paginated details table, and data refresh metadata. Define each KPI in a data dictionary, link all visualizations to the same filter state, and implement a visible reset. Validate totals against a separate summary calculation and show a deliberate empty result when filters match no rows.

## Review questions and practice

1. Why should a dashboard start from user decisions rather than the chart types available?
2. Why are row count and unique order count different?
3. How should a failed refresh affect the last successful view?
4. What should happen when a user drills down into one chart category?

## Further reading

- Visualization principles: https://matplotlib.org/stable/tutorials/
- pandas groupby: https://pandas.pydata.org/docs/user_guide/groupby.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.